# Threading
**1. Python Concurrency**
* A *process* is a computer program.
* A *thread* refers to a thread of execution within a computer program.

Each program is a process that has at least one thread that executes instructions for that process.

* *Concurrency* refers to executing tasks out of order.
* *Parallelism* refers to executing tasks simultaneously.

When we are developing code, we can achieve concurrency with or without parallelism, although concurrency (e.g., task order being irrelevant) is a prerequesite for parallelism.
Our goal is to speed-up a program by executing two or more tasks simultaneously. We are almost always interested in paralellism when we talk about Python concurrency.

**2. Thread**

Thread-base concurrency is provided via the `threading` module and the `Thread` class.

When to use it? Use it for blocking IO-bound tasks which involve calling instructions in our operating system (the kernel), which will wait for the operation to complete.
* Reading or writing a file from the hard drive.
* Reading or writing to standard output, input, or error (`stdin`, `stdout`, `stderr`).
* Printing a document.
* Socket connections.
* Downloading or uploading a file.
* Querying a server.
* Querying a database.
* Taking a photo or recording a video.

Exception: One execption are those tasks that execute code in a third-party library that explicitly releases GIL.
* When calculating cryptographic hashes using the `hashlib` library.
* When calculating matrix operations using the `Numpy` library from the `SciPy` suite.
* Whe compressing data or files (e.g., `zlib`) and when working with video and image data (e.g., `OpenCV`).


Limitations: A key limitation of `Thread` for thread-based concurrency is that it is subject to the Global Intepreter Lock (GIL). This means that only one thread can run at a time in a Python process, unless the GIL is released, such as during blocking I/O or explicitly in a third-party libraries. This limitation means that although threads can achieve concurrency (executing tasks out of order), it can only achieve parallelism (executing tasks simultaneously) under specific circumstances.

**3. Processes**

Process-based concurrency is provided via the `multiprocessing` module and the `Process` class.

When to use it? Use it for CPU-bound tasks which involve performing a computation and does not involve IO, and where data needs to be shared between processes. They typically only involve data in main memory (RAM) or cache and performing computations on or with that data. 
* Calculating points in a fractal.
* Estimating PI.
* Facoting primes.
* Parsing HTML, JSON, etc. documents.
* Processing text.
* Running simulations.

Limitations: A key limitation of the `multiprocessing` module is sharing data between processes. Unlike threads that have shared memory, processes must share data and variables using Inter-Process Communitation (IPC). This requires that data be serialized in order to be transmitted to another process, and deserialized once received. Python-native serialization in the `pickle` module is used and performed automatically. Data is shared using pipes or queues.

Why note always use `multiprocessing`?
1. Heavyweight: Processes are heavyweight structures making them slower to start and require more memory.
2. IPC: All data sent between processes must be serialized, adding a computational overhead proportional to the data shared.
3. Limited Number: The operating system imposes limits on the number of child processes we can create.

**4. Global Interpreter Lock (GIL)**

Python uses GIL to make instructions executed by the Python interpreter thread-safe. It uses a synchronization primitive called a mutual exclusion or a mutex lock to ensure that only one thread of execution can execute instructions at a time within a Python process.

The effect of the GIL is that whenever a thread within a Python program wants to run, it must acquire the lock before executing. This is not a problem for most python programs that have a single thread of execution, called the main thread. It can become a problem in multi-headed Python programs.

The lock is also released in certain situations, allowing other threads to execute. This can happen, for example, during blocking I/O operations or when third-party Python libraries perform computationally intensive tasks in C code (e.g., array operations in NumPy).

## Create & Start New Threads
* *Main Thread*: Default thread created by a main process in Python program, has the name *MainThread*. 
* *Helper Threads*: There may be other threads in our program, created by the standard library or third-party libraries, that assist a primary activity.
* *New Threads*: New threads created directly in our program.
* *Worker Threads*: We may create a pool of reusable worker threads, called a thread pool. We do not create threads in the pool directly, they are created for us by the pool. They are a specific type of helper thread commonly referred to as thread worker, worker threads, or simply workers.

In [8]:
from time import sleep
from random import random
from threading import Thread, local, current_thread, main_thread, Lock, Semaphore, Event, Condition, Barrier
import threading
from queue import Queue

In [2]:
# custom function to be executed in a new thread
def task():
    # block for a moment
    sleep(1)
    # report a message
    print('This is from another thread')
    
# protect the entry point
if __name__ == '__main__':
    # create a new thread instance
    thread = Thread(target=task)
    # This does not start the thread immediately, but instead allows the operating system to schedule the function
    # to execute as soon as possible.
    thread.start()
    # wait for the thread to finish.
    print('Waiting for the thread...')
    # explicitly block and wait for the new thread to terminate.
    thread.join()

Waiting for the thread...
This is from another thread


Extending the `thread` class

In [3]:
# custom thread class
class CustomThread(Thread):
    # override the run function
    def run(self):
    # block for a moment
        sleep(1)
        # report a message
        print('This is another thread')

# protect the entry point
if __name__ == '__main__':
    # create the thread. 
    thread = CustomThread()
    # The inherited `start()` method is then called which starts a new thread and executes the content of the `run()`
    # method in the new thread.
    thread.start()
    # wait for the thread to finish
    print('Waiting for the thread to finish')
    thread.join()

Waiting for the thread to finish
This is another thread


**Thread-Local Storage**

It is a mechanism in multithreaded programming that allows data to be stored and accessed in a way that is private to each thread.

Thread-local storage means that every thread in a program has its own personal space to store data. Even if all threads use the same variable names, like `address`, each thread keeps its own separate copy of that variable. This means that one thread can change its address without affecting another thread’s address. It’s like each worker in a team having their own notebook with the same section titles, but each worker writes their own information inside. This is useful when many threads are doing the same task at the same time and need to use the same code, but each one must keep track of its own data without mixing it up with the others.

In [5]:
# custom function executed in a new thread
def task(shared_local):
    # block for a moment to simulate work
    sleep(1)
    # store a private variable on the thread local
    shared_local.value = 33
    # report the stored value
    print(f'Thread stored: {shared_local.value}')
    
# protect the entry point
if __name__ == '__main__':
    # create a shared thread-local instance
    local_storage = local()
    # store a private variable on the thread local
    local_storage.value = 100
    # report the stored value
    print(f'Main stored: {local_storage.value}')
    # create a new thread to run the custom function
    thread = Thread(target=task, args=(local_storage,))
    # start the new thread
    thread.start()
    # wait for the thread to terminate
    thread.join()
    # report the stored value
    print(f'Main sees: {local_storage.value}')

Main stored: 100
Thread stored: 33
Main sees: 100


## Configuring & Interacting with Threads

**1. Configure Thread Name**

Assigning custom names

In [6]:
# protect the entry point
if __name__ == '__main__':
    # create a thread with a custom name
    thread = Thread(name='MyThread')
    # report thread name
    print(thread.name)

MyThread


**2. Configure Daemon Threads**

Threads can be configured to be *daemon* or *daemonic*, that is, they can be configured as backgrounds threads. The main thread can only exit once all non-daemon threads have exited.

In [7]:
# protect the entry point
if __name__ == '__main__':
    # create a daemon thread
    thread = Thread(daemon=True)
    # report if the thread is a daemon
    print(thread.daemon)

True


**3. Query Thread Native Identifier**

Each thread has a unique identifier assigned by the Python interpreter and a unique native thread identifier (TID), assigned by the operating system.

In [8]:
# protect the entry point
if __name__ == '__main__':
    # create a thread
    thread = Thread()
    # report the thread identifier
    print(thread.native_id)
    # start the thread
    thread.start()
    # report the thread identifier
    print(thread.native_id)

None
1944


**4. Query if Thread is Alive**

A `Thread` object can be alive (running) or dead (terminated). If a thread is alive, it means that the `run()` method of the `Thread` object is currently executing. Thus means that before the `start()` method is called and after the `run()` method has completed, the thread will not be alive. 

In [10]:
# protect the entry point
if __name__ == '__main__':
    # create the thread
    thread = Thread()
    # report the thread is alive
    print(thread.is_alive())

False


**5. Get the Current Thread**

We can get a `Thread` object for the thread running the current code.

In [12]:
# protect the entry point
if __name__ == '__main__':
    # get the current thread
    thread = current_thread()
    # report details
    print(thread)

<_MainThread(MainThread, started 18796)>


**6. Get the Main Thread**

In [14]:
# protect the entry point
if __name__ == '__main__':
    # get the main thread
    thread = main_thread()
    # report details
    print(thread)

<_MainThread(MainThread, started 18796)>


**7. Get all active Threads**

`enumerate()` module function that returns an interable of `Thread` objects, one for each running thread.

In [19]:
# custom function to be executed in a new thread
def task():
    # block for a moment
    sleep(1)

# protect the entry point
if __name__ == '__main__':
    # create a number of new threads
    threads = [Thread(target=task) for _ in range(5)]
    # start the new threads
    for thread in threads:
        thread.start()
    # get a list of all running threads
    running_threads = threading.enumerate()
    # report a count of active threads
    print(f'Active Threads: {len(running_threads)}')
    # report each in turn
    for thread in running_threads:
        print(thread)

Active Threads: 11
<_MainThread(MainThread, started 18796)>
<Thread(Thread-6 (_thread_main), started daemon 1984)>
<Heartbeat(Thread-7, started daemon 11148)>
<ControlThread(Thread-5, started daemon 12704)>
<HistorySavingThread(IPythonHistorySavingThread, started 3360)>
<ParentPollerWindows(Thread-4, started daemon 10904)>
<Thread(Thread-20 (task), started 2816)>
<Thread(Thread-21 (task), started 9460)>
<Thread(Thread-22 (task), started 10604)>
<Thread(Thread-23 (task), started 18328)>
<Thread(Thread-24 (task), started 1624)>


**8. Handle Unexpected Exceptions in New Threads**

Ideally, we would like to know when an unexpected exception occurs in new threads so that we might take appropriate action to clean-up any resources and perhaps log the fault.

We can specify a custom exception hook function that will be called whenever a `Thread` fails with an unhandled `Error` or `Exception` using `excepthook`.

In [20]:
# Register a custom function to handle an exception raised in new threads.

# custom exception hook
def custom_hook(args):
    # report the failure
    print(f"Thread failed: {args.exc_value}")
    
# target function that raises an exception
def task():
    # report a message
    print("Working...")
    # block for a moment
    sleep(1)
    # rise an "unexpected" exception
    raise Exception("Something bad happened")
    
# protect the entry point
if __name__ == "__main__":
    # register the exception hook function
    threading.excepthook = custom_hook
    # create a thread
    thread = Thread(target=task)
    # run the thread
    thread.start()
    # wait for the thread to finish
    thread.join()
    # report that the main thread is not dead
    print("Continuing on ...")

Working...
Thread failed: Something bad happened
Continuing on ...


## Synchronize & Coordinate Threads
**1. Protect Critical Sections with a Mutex Lock**

A mutual exclusion lock or mutex lock is a concurrency primitive intended to prevent a race condition.

A race condition is a concurrency failure case when 2 threads run the same code and access or update the same resource (e.g., data variables, stream, etc.) leaving the resource in an unknown and incosistent state.

Race conditions often result in unexpected behaviour of a program and/or corrupt data.

In [6]:
# custom function to be executed in a new thread
def task(shared_lock, ident, value):
    # acquire the lock. ensures that only one thread at a time can execute the code inside the with block
    # without a lock, multiple threads could execute the print() and sleep() calls simultaneously
    with shared_lock:
        # report a message
        print(f">{ident} got lock, sleeping {value}")
        # block for a fraction of a second
        sleep(value)
        
# protect the entry point
if __name__ == "__main__":
    # create the shared mutex lock
    lock = Lock()
    # create a number of threads with different tags
    threads = [Thread(target=task, args=(lock, i, random())) for i in range(10)]
    # start the threads
    for thread in threads:
        thread.start()
    # wait for all threads to finish
    for thread in threads:
        thread.join()

>0 got lock, sleeping 0.8922408344177057
>1 got lock, sleeping 0.3944154305653341
>2 got lock, sleeping 0.7638335940807877
>3 got lock, sleeping 0.0662580071981691
>4 got lock, sleeping 0.9657879984203998
>5 got lock, sleeping 0.2795623094541646
>6 got lock, sleeping 0.16094211771842015
>7 got lock, sleeping 0.14016258848185048
>8 got lock, sleeping 0.1798947608360698
>9 got lock, sleeping 0.3944583798985133


The `threading` module also provides a reentrant lock via the `RLock` class. This mutex allows a thread to acquire the lock subsequently multiple times (e.g., is reentrant) unlike the `Lock` class. Reentrant locks are helpful in cases where the same lock may be used in multiple places in the code and a function that makes use of the lock may be called from a critical section where the lock is already held.

**2. Limit Access to a Resource with a Semaphore**

A semaphore is a concurrency primitive that allows a limit on the number of threads that can acquire a lock protecting a critical section or resource. It adds a count for the number of threads that can acquire the lock before additional threads will block. Once full, new threads can only acquire access on the semaphore once an existing thread holding the semaphore release access. If it is set to be 1, then the semaphore will operate like a mutex lock.

In [9]:
# custom function to be executed in a new thread.
def task(shared_semaphore, ident):
    # attempt to acquire the semaphore
    with shared_semaphore:
        # generate a random value between 0 and 1 
        val = random()
        # block for a fraction of a second
        sleep(val)
        # report the result
        print(f"Thread {ident} got {val}")
        
# protect the entry point
if __name__ == "__main__":
    # create the shared semaphore
    semaphore = Semaphore(2)
    # create threads
    threads = [Thread(target=task, args=(semaphore, i)) for i in range(10)]
    # start new threads
    for thread in threads:
        thread.start()
    # wait for new threads to finish
    for thread in threads:
        thread.join()

Thread 0 got 0.2797088076187305
Thread 1 got 0.6185897728394737
Thread 2 got 0.8935508091164953
Thread 3 got 0.6961669802819656
Thread 5 got 0.8505213949364299Thread 4 got 0.9798298232900595

Thread 7 got 0.16271715337209347
Thread 8 got 0.5216402073422705
Thread 6 got 0.7824976766951203
Thread 9 got 0.6587994721756855


All 10 threads attempt to acquire the semaphore, but only 2 threads are granted access at a time. The threads on the semaphore do their work and release the semaphore when they are done, at random intervals.

**3. Signal Between Threads Using an Event**

An event is a thread-safe boolean flag that can be used to signal between two or more threads.

Threads sharing the `Event` object can check if the event is set, set the event, clear the event (make it not set), or wait for the event to be set.

Threads can wait for the event to set via the `wait()` method. Calling this method will block until the event is marked as set (e.g., another thread calling the `set()` method). If the event is already set, the `wait()` method will return immediately.

In [3]:
# custom function to be executed in a new thread
def task(shared_event, number):
    # wait for the event to be set
    print(f"Thread {number} waiting...")
    shared_event.wait()
    # begin work, generate a random number
    value = random()
    # block for a fraction of a second
    sleep(value)
    # report a message
    print(f"Thread {number} got {value}")
    
# protect the entry point
if __name__ == "__main__":
    # create a shared event object
    event = Event()
    # create a suit of threads
    threads = [Thread(target=task, args=(event, i)) for i in range(5)]
    # start all threads
    for thread in threads:
        thread.start()
    # block thread a moment
    print("Main  thread blocking...")
    sleep(2)
    # trigger all threads
    event.set()
    # wait for all threads to terminate
    for thread in threads:
        thread.join()        

Thread 0 waiting...
Thread 1 waiting...
Thread 2 waiting...
Thread 3 waiting...
Thread 4 waiting...
Main  thread blocking...
Thread 3 got 0.05818944048304053
Thread 2 got 0.347752142669857
Thread 4 got 0.5529878835366032
Thread 1 got 0.6503940000849743
Thread 0 got 0.7252285459885333


**4. Coordinate Using a Condition Variable**

A condition variable (also called monitor) allows multiple threads to wait and be notified about some result. 

A condition can be acquired by a thread after which it can wait to be notified by another thread that something has changed. While waiting, the thread is blocked and releases the lock on the condition for other threads to acquire.

Another thread can then acquire the condition, make a change in the program, and notify one, all, or a subset of threads waiting on the condition that something has changed.

The waiting thread can then wake-up, re-acquire the condition, perform checks on any changed state and perform required actions.

In [7]:
# custom function to be executed in a new thread
def task(shared_condition):
    # block for a moment
    sleep(1)
    # notify a waiting thread that the work is done
    print("Thread sending notification...")
    with shared_condition:
        # notify a waiting threads. the notified threads will stop-blocking as soon as it can reacquire the condition.
        # this is like ringing the bell which wakes up the main thread
        shared_condition.notify()
        
# protect the entry point
if __name__ == "__main__":
    # create a condition. this is the shared "doorbell" between threads.
    condition = Condition()
    print("Main thread waiting for data...")
    # acquire the condition. `with` will acquire/release the condition. this is like creating the doorbell
    with condition:
        # create a new thread to execute the task
        thread = Thread(target=task, args=(condition, ))
        # start the new thread
        thread.start()
        # wait to be notified by the new thread. We can also pass a `timeout` argument which will allow the thread
        # to stop blocking after a time limit in seconds. here the main threat waits
        condition.wait()
    # we know the data is ready
    print("Main thread all done")

Main thread waiting for data...
Thread sending notification...
Main thread all done


**5. Coordinate Threads with a Barrier**

A barrier is a synchronization primitive.

It allows multiple threads to wait on the same barrier object until a predefined fixed number of threads arrive (e.g., the barrier is full), after which all threads are then notified and released to continue their execution.

Internally, a barrier maintains a count of the number of threads waiting on the barrier and a configured maximum number of parties (threads) that are expected. Once the expected number of parties reaches the pre-defined maximum, all waiting threads are notified. 

```python
barrier = Barrier(10, action=my_function)
```

In [10]:
# custom function to be executed in a new thread 
def task(shared_barrier, ident):
    # generate a unique value between 0 and 10
    value = random() * 10
    # block for a moment
    sleep(value)
    # report result
    print(f"Thread {ident} got: {value}")
    # wait for all other threads to complete
    shared_barrier.wait()
    
# protect the entry point
if __name__ == "__main__":
    # create a barrier for (5 threads + 1 main thread)
    barrier = Barrier(5 + 1)
    # create the new threads
    threads = [Thread(target=task, args=(barrier, i)) for i in range(5)]
    # start the new threads
    for thread in threads:
        # start thread
        thread.start()
    # wait for all new threads to finish
    print("Main thread waiting on all results ...")
    barrier.wait()
    # report once all thread are done
    print("All threads have their result")

Main thread waiting on all results ...
Thread 2 got: 0.5071185329852379
Thread 3 got: 1.159509342413284
Thread 4 got: 6.502711268165723
Thread 0 got: 7.8411178212087815
Thread 1 got: 9.742375189984294
All threads have their result


Imagine you have several threads (like small workers) running in parallel, each doing some work.
At some point, you want all of them to stop and wait until everyone reaches the same point — and then continue together.

## Share Data Between Threads
**1. Access Global Variables From Multiple Threats**

Threads can use global variables to share data between threads. Threads are able to share memory within a process, such as variables.

We can protect the global variable from race conditions by using a mutual exclusion lock via the `Lock` class. Each time the global variable is read or modified, it must be done so via the `Lock`. Specifically, we must acquire the `Lock` before we attempt to interact with the global variable. This will ensure that only one thread is interacting with the global variable at the time. 

In [4]:
# define a global variable
data = 66
# define a lock to protect the global variable
lock = Lock()

# custom function
def custom():
    # acquire the lock for the global variable
    with lock:
        # modify the global variable
        data = 33

**2. Return Values from Threats via Instance Variables**

When using new threads, we may need to return a value from a thread to another thread, such as the main thread. Reasons could be:
* The new thread loaded some data that needs to be returned. 
* The new thread calculated something that needs to be returned.
* The new thread needs to be share its state or status with another thread.

The problem is we cannot return values from threads.

`start()` method does not block, instead it returns immediately and does not return a value. `run()`, `join()` also do not return values.

A straightforward approach to returning values from a new thread is to extend the `Thread` class and store return values in instance varibles.

In [5]:
# custom thread
class CustomThread(Thread):
    
    # constructor
    def __init__(self):
        # execute the base constructor
        Thread.__init__(self)
        # set a default value
        self.value = None
        
    # function executed in a new thread
    def run(self):
        # block for a moment
        sleep(1)
        # store data in an instance variable
        self.value = "Hello from a new thread"
        
# protect the entry point
if __name__ == "__main__":
    # create a new thread
    thread = CustomThread()
    # start the thread
    thread.start()
    # wait for the thread to finish
    thread.join()
    # get the value returned from the thread
    data = thread.value
    print(data)

Hello from a new thread


**3. Share Data Between Threads with Queues**

A queue is a thread-safe data structure that can be used to share data between threads without a race condition. The **queue** has the following classes:
* `Queue`: A fully-featured first-in-first-out (FIFO) queue.
* `SimpleQueue`: A FIFO queue with less functionality.
* `LifoQueue`: A last-in-first-out (LIFO) queue.
* `PriorityQueue`: A queue where the first items out are those with the highest priority.

Items can be added by a call to `put()` and retrieved by a call to `get()`.

```python
# created a queue with a maximum capacity
queue = Queue(maxsize=100)
```

Example of sharing data between 2 threads using a queue.

In [2]:
# custom function to be executed in a new thread
def task(shared_queue):
    """
    Takes the queue as an argument, generates some data and puts that data on the queue. It is a producer thread.
    """
    # block for a moment
    sleep(1)
    # prepare some data
    item = "Hello from a new thread"
    # share the data via the queue
    shared_queue.put(item)
    
# protect the entry point
if __name__ == "__main__":
    # create the shared queue
    queue = Queue()
    # create a new thread
    thread = Thread(target=task, args=(queue,))
    # start the new thread
    thread.start()
    # block and wait for data via queue
    data = queue.get()
    # report data from the queue
    print(data)

Hello from a new thread
